# Train + evaluate the image captioning model on a free Colab GPU

Trains the CNN (InceptionV3) + LSTM captioner on Flickr8k, then scores it with BLEU-1..4 on the
validation split `train.py` holds out (seed 42, 10%, 809 images) and renders a grid of sample captions.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Expect roughly 1–2 hours for 20 epochs.
Keep the tab open. Checkpoints are written to your Google Drive, so a disconnect does not lose them.
At the end, download `results.zip` and put its contents in the repo's `results/` folder.

In [ ]:
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (Runtime -> Change runtime type)"
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/lalith0087/IMAGE-CAPTIONING-using-CNN-and-LSTM.git
%cd IMAGE-CAPTIONING-using-CNN-and-LSTM
!pip -q install nltk tqdm
!ls scripts

## 1. Data (public Flickr8k mirror, ~1.1 GB)

In [ ]:
import glob, os, shutil
!mkdir -p data
!wget -q -O /tmp/f8k_img.zip https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -q -O /tmp/f8k_txt.zip https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
!unzip -q -o /tmp/f8k_img.zip -d /tmp/f8k_img
!unzip -q -o /tmp/f8k_txt.zip -d /tmp/f8k_txt

# the archive spells the image folder "Flicker8k_Dataset"; find whichever folder holds the jpgs
img_dir = max((os.path.dirname(p) for p in glob.glob("/tmp/f8k_img/**/*.jpg", recursive=True)),
              key=lambda d: len(os.listdir(d)))
shutil.rmtree("data/Flickr8k_Dataset", ignore_errors=True)
shutil.move(img_dir, "data/Flickr8k_Dataset")
shutil.copy(glob.glob("/tmp/f8k_txt/**/Flickr8k.token.txt", recursive=True)[0], "data/Flickr8k.token.txt")
print(len(os.listdir("data/Flickr8k_Dataset")), "images")
!wc -l data/Flickr8k.token.txt   # expect 40460

In [ ]:
!python scripts/build_vocab.py

## 2. Keep checkpoints on Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/captioning_checkpoints
!rm -rf checkpoints && ln -s /content/drive/MyDrive/captioning_checkpoints checkpoints
!ls -la checkpoints

## 3. Train (change `EPOCHS` if you like; the README's default is 20)

In [ ]:
EPOCHS = 20
!python -u scripts/train.py --epochs {EPOCHS} --batch-size 64

## 4. Evaluate on the held-out split

In [ ]:
!python scripts/evaluate.py --batch-size 64 --workers 2

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open("results/metrics.json")), indent=2))
display(Image("results/samples.png"))

## 5. Download the results

`results.zip` has `metrics.json`, `captions.json` and `samples.png`. The checkpoint stays in your Drive (it is ~118 MB, over GitHub's 100 MB file limit).

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")